# Chuẩn bị đầu vào Milestone 2 từ backup Milestone 1

Notebook khôi phục source LaTeX/BibTeX, metadata và reference metadata sang một thư mục **preparationM2 riêng trên Drive**. Đây là bước chuẩn bị raw input; parser, hierarchy, cleanup và dedup M2 sẽ được viết/chạy sau.

**M1 backup → kiểm SHA-256/receipts → raw_m1 + inventory → parser M2 về sau.**

Dùng thư mục backup Drive nguyên vẹn hoặc ZIP tải từ Drive. Notebook nhận cả checkpoint/full backup/session report, không nhầm manifest-only với source thật. Chạy **CPU / accelerator None**, một runtime cho một DATASET_NAME; không chạy hai máy ghi cùng đích.

## 1. Đường dẫn của bạn

Trên Colab, Drive được mount ở bước 2. Sửa SOURCE_PATH theo thư mục backup M1 thật; mặc định là đường dẫn crawler M1 đã dùng. Nếu dùng ZIP, trỏ trực tiếp tới file ZIP trên Drive hoặc local, không cần giải nén.

DATASET_NAME phân biệt tài khoản, ví dụ Danh, Danh1, Danh2, Khanh1. ZIP đã kiểm tra trong phiên phát triển có queue Danh2; đó chỉ là dữ liệu một tài khoản. Cả nhóm có thể giữ các dataset riêng dưới cùng preparationM2/raw_m1, hoặc trỏ SOURCE_PATH tới thư mục cha chứa đủ backup đã gom.

Khi mở notebook từ GitHub trên Colab, repo phải có src/preparation/milestone2.py mới. Commit/push code mới trước khi clone, hoặc đưa checkout mới lên runtime. Không cần cài toàn bộ dependencies của crawler cho bước khôi phục này.
Mặc định mỗi lượt chuẩn bị **100 bài (ID)**, bao gồm tất cả version của mỗi bài. Đổi `MAX_PAPERS_PER_RUN` rồi chạy lại các cell sẽ tự chọn những ID chưa xử lý; giữ nguyên `SOURCE_PATH`, `PREPARATION_DIR` và `DATASET_NAME`. Thứ tự ổn định theo đường dẫn backup rồi ID trong manifest, không phải thứ tự số ID toàn corpus. Không đổi tên dataset giữa các lượt nếu muốn tiếp tục cùng tiến độ.

`RETRY_BLOCKED=False` giúp ID lỗi không chiếm lại lượt và cản ID mới. Bật True để thử lại các ID lỗi (vẫn tính trong giới hạn mỗi lượt). `MAX_PAPERS_PER_RUN=None` kiểm/khôi phục lại toàn corpus, dùng khi cần sửa dữ liệu đã lưu bị hỏng. Nếu thêm/đổi backup gốc, dùng dataset mới hoặc chạy None để xây lại và cập nhật tiến độ; không dùng tiến độ cũ cho một nguồn đã thay đổi.


In [ ]:
from pathlib import Path

SOURCE_PATH = Path('/content/drive/MyDrive/arXivProject/Milestone1')
PREPARATION_DIR = Path('/content/drive/MyDrive/arXivProject/preparationM2')
DATASET_NAME = 'Danh2'  # Đổi đúng tài khoản/dataset của bạn.
PROJECT_DIR = None     # Local: tự tìm repo; có thể đặt Path('/content/arXivProject').
REPOSITORY_URL = 'https://github.com/Hireophant/arXivProject.git'
REPOSITORY_BRANCH = 'master'
CLONE_IF_NEEDED = True
SHOW_PROGRESS_EVERY = 10
MAX_PAPERS_PER_RUN = 100  # Số ID mới mỗi lượt, gồm mọi version của bài; None = làm lại toàn bộ.
RETRY_BLOCKED = False      # True: cho phép chọn lại ID đã làm nhưng bị blocked_input.


## 2. Mở source code và mount Drive

Chỉ clone khi chưa có checkout và đang chạy Colab; không tự pull/ghi đè code đang có. Notebook không gọi arXiv hoặc Semantic Scholar. Quyền Drive chỉ được xin qua hộp thoại mount của Colab khi chính bạn chạy cell này.

In [ ]:
import importlib.util
import json
import os
import re
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python >= 3.11.')
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ModuleNotFoundError, ValueError):
    IN_COLAB = False

if PROJECT_DIR is None:
    PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'src/preparation/milestone2.py').is_file()), None)
if PROJECT_DIR is None and IN_COLAB:
    PROJECT_DIR = Path('/content/arXivProject')
if PROJECT_DIR is None:
    raise RuntimeError('Mở notebook trong checkout mới hoặc đặt PROJECT_DIR tới project root.')
PROJECT_DIR = Path(PROJECT_DIR).resolve()
if not PROJECT_DIR.exists() and IN_COLAB and CLONE_IF_NEEDED:
    subprocess.run(['git', 'clone', '--branch', REPOSITORY_BRANCH,
                    REPOSITORY_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'src/preparation/milestone2.py').is_file():
    raise RuntimeError('Checkout chưa có preparation helper mới. Đưa code mới lên runtime rồi chạy lại.')
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from preparation import inspect_backups, prepare_corpus, validate_prepared_corpus

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', DATASET_NAME):
    raise ValueError('DATASET_NAME dùng chữ không dấu/số/_/-, ví dụ Danh2 hoặc Khanh1.')
OUTPUT_ROOT = Path(PREPARATION_DIR) / 'raw_m1' / DATASET_NAME
print('Python:', sys.version.split()[0])
print('Source:', SOURCE_PATH)
print('Đích raw M2:', OUTPUT_ROOT)

## 3. Kiểm kê backup trước khi khôi phục

Preview chỉ đếm receipt và đọc tên queue, **chưa xác nhận source đầy đủ**. Không chọn riêng report.zip hay một corpus.zip bên trong backup; chọn cả thư mục Milestone1 hoặc ZIP tải từ Drive chứa thư mục đó. Giữ backup_receipt.json cạnh corpus.zip.

In [ ]:
preview = inspect_backups(SOURCE_PATH)
print('Loại input:', preview['source_kind'])
print('Số bộ backup:', preview['backup_count'])
print('Corpus ZIP thiếu receipt:', len(preview['orphan_corpus_archives']))
print('Queues/tài khoản:', json.dumps(preview['queues'], ensure_ascii=False, indent=2))
if not preview['backup_count']:
    raise ValueError('Không tìm thấy backup_receipt.json; kiểm tra lại SOURCE_PATH.')
print('Preview chưa xác minh hash. Cell tiếp theo sẽ kiểm và khôi phục.')

## 4. Kiểm hash và khôi phục raw input

Mỗi corpus.zip phải khớp hash/size trong backup receipt; source từng version phải khớp M1 source receipt. Source giữ nguyên bytes, đường dẫn file con và citations. Metadata/references được lưu theo cấu trúc JSON dễ dùng lại.

Chạy lại cùng đích sẽ kiểm lại dữ liệu và phục hồi file đã hỏng; không nhân đôi publication/version. Nếu runtime ngắt, preparation.json còn in_progress: chạy lại để hoàn thành, không dùng kết quả đang dở. Nếu hai bản source hợp lệ của cùng version có nội dung khác nhau, version bị chặn để kiểm thay vì chọn ngẫu nhiên.

Không dùng thư mục đích có file cá nhân sẵn. Notebook chỉ tiếp tục đích đã do chính helper này tạo; M1 và M2 phải là hai cây không chồng nhau.
Sau khi lưu, helper đọc lại source/metadata/references và inventory, ghi input_validation.json + snapshot_id. Kiểm tra không đạt sẽ chặn bàn giao. Một snapshot valid có thể còn version blocked; đó là input partial đã kiểm, không phải toàn corpus đã đủ.

Tiến độ nằm trong `batch_checkpoint.json`, chỉ chốt sau khi đọc lại và kiểm tra output thành công. Runtime ngắt giữa lượt: chạy lại sẽ chọn lại lượt chưa chốt, tận dụng source hợp lệ đã ghi; không nhảy mất ID. Mỗi lượt vẫn đọc/kiểm hash các archive và kiểm lại phần output đã chuẩn bị, nên giới hạn 100 bài không có nghĩa thời gian giảm còn đúng 100/tổng số bài.

ID chưa tới lượt vẫn được ghi vào manifest/inventory với `blocked_input`, reason `PENDING_BATCH`, không có đường dẫn source dùng cho parser. Các ID đã thử nhưng thiếu/hỏng source có reason lỗi riêng và được tính là đã xử lý, chưa chắc đã sẵn sàng. Tiếp tục chạy đến khi số bài chờ bằng 0; nếu còn lỗi input thì vẫn là partial.


In [ ]:
progress_count = 0
def show_progress(message):
    global progress_count
    progress_count += 1
    if not message.startswith('[') or progress_count == 1 or progress_count % max(1, int(SHOW_PROGRESS_EVERY)) == 0:
        print(message, flush=True)

summary = prepare_corpus(SOURCE_PATH, OUTPUT_ROOT, progress=show_progress,
                         max_papers=MAX_PAPERS_PER_RUN, retry_blocked=RETRY_BLOCKED)
print(json.dumps(summary, ensure_ascii=False, indent=2))
if not summary.get('input_valid'):
    raise RuntimeError('Dữ liệu sau lưu không đạt kiểm tra. Xem input_validation.json rồi chạy lại bước chuẩn bị.')
batch = summary['batch']
print(f"Lượt này: {len(batch['selected_ids'])} bài; đã xử lý: {batch['papers_processed']}/{summary['paper_count']}; còn chờ: {batch['papers_pending']}.")
print('ID lượt này:', batch['selected_ids'])
if batch['papers_pending']:
    print('CHƯA HẾT CÁC LƯỢT: chạy lại cell khôi phục để lấy những ID chưa xử lý tiếp theo.')
if summary['status'] == 'ready':
    print('RAW INPUT READY: mọi version phát hiện đã có source hợp lệ hoặc no_tex có bằng chứng.')
else:
    print('RAW INPUT CHƯA ĐỦ: xem blocked_versions.csv và backup_audit.jsonl, giữ nguyên trạng thái lỗi.')

## 5. Xem bài/version đã chuẩn bị

ready_tex: source có TeX đã kiểm; empty_no_tex: M1 xác nhận không có TeX; blocked_input: thiếu/hỏng hoặc source xung đột. Đây là trạng thái **input**, không phải parser M2 đã chạy thành công. Lỗi Semantic Scholar không tự loại bài có source TeX hợp lệ.
`PENDING_BATCH` là bài chưa tới lượt, khác lỗi source thật. Bảng đếm toàn bộ phạm vi backup, không chỉ lượt hiện tại; `batch.selected_ids` ghi chính xác các ID của lượt này. Snapshot thay đổi khi có thêm bài sẵn sàng, cần lấy lại snapshot mới trước mỗi lượt parse.


In [ ]:
import csv
from html import escape

with (OUTPUT_ROOT / 'inventory.csv').open(encoding='utf-8', newline='') as stream:
    inventory = list(csv.DictReader(stream))
pending = [r for r in inventory if r['reason'] == 'PENDING_BATCH']
blocked = [r for r in inventory if r['status'] == 'blocked_input' and r['reason'] != 'PENDING_BATCH']
def show_rows(rows, limit=20):
    shown = rows[:limit]
    fields = ['arxiv_id', 'version', 'status', 'reason', 'm1_error']
    try:
        from IPython.display import HTML, display
    except ImportError:
        for row in shown:
            print({k: row[k] for k in fields})
        return
    html = '<table><thead><tr>' + ''.join('<th>' + escape(k) + '</th>' for k in fields) + '</tr></thead><tbody>'
    html += ''.join('<tr>' + ''.join('<td>' + escape(str(row[k])) + '</td>' for k in fields) + '</tr>' for row in shown)
    display(HTML(html + '</tbody></table>'))
print(f"{summary['paper_count']} bài; {len(inventory)} versions; {len(blocked)} versions lỗi; {len(pending)} versions chưa tới lượt.")
show_rows(inventory)
if blocked:
    print('Các version bị chặn (tối đa 20 dòng):')
    show_rows(blocked)

## 6. Bàn giao cho phần code parser M2

Cấu trúc đã lưu:

```text
preparationM2/raw_m1/<DATASET_NAME>/
  manifest.json                  # schema m2-input-1.0, papers + expected versions + statuses
  preparation.json               # in_progress / ready / partial / failed, run_id
  batch_checkpoint.json          # tiến độ ID đã chốt sau validation; giữ nguyên để tiếp tục
  input_validation.json          # kiểm đọc lại, run_id và snapshot_id
  inventory.csv                  # đúng một dòng cho mỗi publication/version
  blocked_versions.csv
  backup_audit.jsonl              # hash archive và các lỗi input
  provenance.jsonl                # backup/fingerprint của từng source copy
  <paper_key>/
    metadata.json
    reference_result.json        # kết quả references M1 đầy đủ đã chọn
    references.json
    references_all.json
    sources/.vN.receipt.json
    sources/vN/**/*.tex, **/*.bib
```

Parser sau này đọc manifest và các source_dir có trạng thái phù hợp; no_tex không sinh fake elements, blocked vẫn giữ trong coverage. Schema chuẩn bị này khác schema checkpoint crawler 3.0: không đưa lại vào crawler như một run M1 và không sửa source ở đây. Các bảng elements/hierarchy/bibtex M2 nên lưu dưới `Milestone2/parsed/<DATASET_NAME>/` (thư mục kết quả riêng, ngoài preparationM2).

Một người có thể code parser trên mẫu raw đã kiểm ngay; muốn xác nhận bao phủ toàn nhóm phải chuẩn bị đủ dataset của bốn người/12 accounts rồi đối chiếu inventory chung. Không cộng checkpoint/retry thành bài mới. Nếu thiếu backup, lấy lại đúng ZIP/receipt gốc; notebook không tự crawl bù.
Giữ preparationM2 chỉ để đọc khi làm parser. Chạy lại validate_prepared_corpus(RAW_CORPUS_ROOT) trước mỗi lượt parse; không chỉ tin một file marker cũ sau khi source đã đổi. Nhóm ghi snapshot_id vào config/log/test evidence để cùng đối chiếu một bộ raw input.

Khi còn `PENDING_BATCH`, raw snapshot chỉ sẵn sàng để phát triển/thử parser trên phần đã có; chưa được báo là toàn bộ corpus đã đủ. Parser phải giữ các version chờ ở coverage dưới `blocked_input`, reason `PENDING_BATCH`, không đọc metadata/source chưa được khôi phục và không bỏ ID khỏi mẫu số metrics. Record/metadata trong manifest của ID chờ chỉ mô tả phạm vi backup, không xác nhận file metadata đã được ghi. Bước đóng gói/chuyển raw cũng giữ nguyên manifest và inventory này.


In [ ]:
prepared_manifest = json.loads((OUTPUT_ROOT / 'manifest.json').read_text(encoding='utf-8'))
completion = json.loads((OUTPUT_ROOT / 'preparation.json').read_text(encoding='utf-8'))
if completion['run_id'] != prepared_manifest['run_id'] or completion['status'] == 'in_progress':
    raise RuntimeError('Output chưa hoàn tất; chạy lại cell khôi phục trước khi bàn giao.')
input_check = validate_prepared_corpus(OUTPUT_ROOT)
saved_check = json.loads((OUTPUT_ROOT / 'input_validation.json').read_text(encoding='utf-8'))
if (not input_check['valid'] or not saved_check.get('valid') or saved_check['run_id'] != completion['run_id']
        or input_check['snapshot_id'] != saved_check.get('snapshot_id')
        or input_check['snapshot_id'] != completion.get('snapshot_id')):
    raise RuntimeError('Input hoặc validation receipt đã đổi; chưa được bàn giao. Chạy lại cell khôi phục.')
print('Snapshot đã kiểm:', input_check['snapshot_id'])
print('Toàn phạm vi đã đủ:', input_check['complete'])
print('Bài còn chờ lượt sau:', summary['batch']['papers_pending'])
if summary['batch']['papers_pending']:
    print('BÀN GIAO PARTIAL: chưa đủ toàn corpus; tiếp tục khôi phục trước khi chốt kết quả toàn bộ.')
assert len(inventory) == summary['versions_ready_tex'] + summary['versions_empty_no_tex'] + summary['versions_blocked']
RAW_CORPUS_ROOT = OUTPUT_ROOT
print('RAW_CORPUS_ROOT =', RAW_CORPUS_ROOT)
print('Manifest:', RAW_CORPUS_ROOT / 'manifest.json')
print('Input status:', completion['status'])
print('Bước tiếp theo: viết adapter input và parser theo kế hoạch M2; chưa có kết quả parse trong notebook này.')